# Phase 2a — Baseline 2D-slice CFD for rib-TO aero loads

**Colab-only, CPU runtime** (Runtime -> Change runtime type -> CPU). SU2 is a
single-threaded CPU solver; GPU is wasted quota.

Validates the CFD-slice pipeline (mesh_2d_slice -> cfg -> SU2 -> parsers ->
j_fan) and produces the baseline aero load for `run_phase2_to.py --pressure-pa`.
Thin orchestration — all logic is in tested library code (CLAUDE.md 6). The rib
SIMP TO itself runs locally on analytic loads and does NOT need this notebook.


## 1. Clone repo + Python/native deps

In [ ]:
import os, subprocess, urllib.request, shutil, sys
from pathlib import Path

REPO_DIR = Path('/content/fan-optimization')
if not REPO_DIR.exists():
    !git clone -b phase-2-rib-to https://github.com/clingergab/fan-optimization.git {REPO_DIR}
%cd {REPO_DIR}
sys.path.insert(0, str(REPO_DIR / 'scripts'))

# gmsh dlopens libGLU + X libs at import; Colab CPU runtimes lack them by default.
!apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 libxft2 libxinerama1 unzip
!pip install --quiet -e {REPO_DIR}
!pip install --quiet gmsh numpy scipy jinja2 pandas
print('[deps] installed')


## 2. Install SU2 8.0.1 (pre-built binary — the proven 0.6c path)

In [ ]:
SU2_VERSION = '8.0.1'
SU2_DIR = Path('/content/su2')
SU2_BIN = SU2_DIR / 'bin' / 'SU2_CFD'

if not SU2_BIN.exists():
    url = (f'https://github.com/su2code/SU2/releases/download/'
           f'v{SU2_VERSION}/SU2-v{SU2_VERSION}-linux64.zip')
    print('[su2] downloading', url)
    urllib.request.urlretrieve(url, '/tmp/su2.zip')
    SU2_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', '/tmp/su2.zip', '-d', str(SU2_DIR)], check=True)

# Locate the real SU2_CFD binary (skip symlinks) and put its dir on PATH.
cands = [p for p in SU2_DIR.rglob('SU2_CFD') if p.is_file() and not p.is_symlink()]
assert cands, 'SU2_CFD not found after extract — try the source-build path in colab_spike_0_6c.ipynb'
bindir = cands[0].parent
for p in bindir.iterdir():
    if p.is_file():
        os.chmod(p, 0o755)
os.environ['SU2_RUN'] = str(bindir)
os.environ['PATH'] = str(bindir) + ':' + os.environ['PATH']
os.environ['PYTHONPATH'] = str(bindir) + ':' + os.environ.get('PYTHONPATH', '')
print('[su2] SU2_CFD ->', shutil.which('SU2_CFD'))
assert shutil.which('SU2_CFD'), 'SU2_CFD still not on PATH'


## 3. Prepare the case (mesh + both stroke cfgs)

In [ ]:
import run_phase2a_baseline_cfd as p2a
WORK = REPO_DIR / 'data' / 'phase2a_baseline_cfd'
manifest = p2a.prepare_baseline_case(WORK, n_blades=5)
manifest


## 4. Run SU2 — capture each stroke's history separately

In [ ]:
def run_su2(cfg: str, history_out: str):
    with open(WORK / (cfg + '.log'), 'w') as f:
        subprocess.run(['SU2_CFD', cfg], cwd=str(WORK), check=True,
                       stdout=f, stderr=subprocess.STDOUT)
    hists = sorted(WORK.glob('history*.csv'), key=lambda p: p.stat().st_mtime)
    assert hists, f'no history.csv after {cfg} — check {cfg}.log'
    hists[-1].rename(WORK / history_out)
    print(cfg, '->', history_out)

run_su2('productive.cfg', 'history_productive.csv')
run_su2('return.cfg', 'history_return.csv')


## 5. Extract the baseline load

In [ ]:
load = p2a.extract_baseline_load(
    WORK / 'history_productive.csv',
    WORK / 'history_return.csv',
)
print(load)
# Then, on your machine / driver:
#   python scripts/run_phase2_to.py --pressure-pa <load['suggested_pressure_pa']>


## If SU2 errors
Paste `data/phase2a_baseline_cfd/productive.cfg.log` (or `return.cfg.log`).
Markers in the mesh are `fan_surface` / `farfield` / `cascade_wall`; the cfg
references all three. If the binary release URL 404s, use the full SU2 install
cell (with the source-build fallback) from `notebooks/colab_spike_0_6c.ipynb`.
